# Project 09 — Partial Pooling (Hierarchical Means)

**Scenario.** A continuous assay readout is measured on many plates/labs/batches, with only a *few* observations per group. We want each group's mean **and** the population they come from — without either pretending the groups are identical (complete pooling) or treating each in isolation (no pooling).

**New skill:** *partial pooling* — shrinkage of each group toward the grand mean, with the amount of shrinkage learned from the data. **Key pitfall:** the natural *centered* parameterization creates **Neal's funnel** and produces divergences when the between-group SD $\tau$ is small. The fix is a *non-centered* parameterization.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

We assume groups are **exchangeable**: their latent means $\theta_j$ are draws from a common population $\text{Normal}(\mu, \tau)$, and within a group observations are $\text{Normal}(\theta_j, \sigma)$.

$$\theta_j \sim \text{Normal}(\mu, \tau), \qquad y_{ij} \sim \text{Normal}(\theta_j, \sigma).$$

**Assumptions made explicit:** (a) groups are exchangeable (no group is special a priori), (b) the population of group means is Normal, (c) within-group noise $\sigma$ is common across groups. We synthesize from known truth ($\mu=5,\ \tau=0.8,\ \sigma=1$) with **few** obs per group so shrinkage is visible and the funnel can bite.

In [ ]:
from data.generate_data import generate
data = generate()
y, group, J = data['y'], data['group'], data['J']
print(f"{J} groups x {data['n_per']} obs = {len(y)} observations")
print(f"true mu={data['truth']['mu']}, tau={data['truth']['tau']}, "
      f"sigma={data['truth']['sigma']}")
emp = np.array([y[group==j].mean() for j in range(J)])
print('per-group empirical means:', np.round(emp, 2))

## Step 2 — Model specification (with justified priors)

Hyperpriors: $\mu \sim \text{Normal}(5, 5)$ (weakly informative, centred near the plausible scale), $\tau \sim \text{HalfNormal}(2)$ and $\sigma \sim \text{HalfNormal}(2)$ (positive scales, gently shrunk).

**Centered vs non-centered.** The centered form $\theta_j \sim \text{Normal}(\mu, \tau)$ couples each $\theta_j$ to $\tau$; when $\tau \to 0$ the joint density narrows into a sharp funnel neck that NUTS cannot traverse at a fixed step size $\Rightarrow$ **divergences**. The non-centered form writes $\theta_j = \mu + \tau\, z_j$ with $z_j \sim \text{Normal}(0,1)$, decoupling the geometry. We fit **non-centered** here and revisit the centered failure in the broken notebook.

In [ ]:
from model import build_model, fit
model = build_model(data, parameterization='noncentered')
model

## Step 3 — Prior predictive checks

We simulate datasets implied by the prior and check that the implied observations cover the scale of the real data without absurd extremes. A HalfNormal(2) on $\tau$ admits both near-complete-pooling ($\tau\approx0$) and substantial spread, which is the flexibility we want the data to resolve.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=400, random_seed=RNG)
pp = prior.prior_predictive['y'].values.ravel()
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(pp, bins=40, color='#55A868', edgecolor='white', density=True)
ax.axvline(y.mean(), color='red', lw=1.5, label='observed mean')
ax.set(xlabel='y implied by prior', ylabel='density',
       title='Prior predictive — covers the data scale')
ax.legend(); plt.tight_layout()

## Step 4 — Inference (NUTS, non-centered)

Settings: `draws=800, tune=1000, chains=4, target_accept=0.9`. Four chains give reliable split-$\hat R$; `target_accept=0.9` is a mild safeguard for hierarchical geometry. The non-centered parameterization should sample cleanly with ~0 divergences.

In [ ]:
idata = fit(data, parameterization='noncentered', draws=800, tune=1000,
            chains=4, target_accept=0.9, seed=101)

## Step 5 — Computational diagnostics

Check $\hat R \approx 1.00$, healthy ESS, and **divergences = 0**. For hierarchical models the **energy plot** (`az.plot_energy`) is essential: a marginal energy distribution that matches the energy-transition distribution indicates NUTS can move through the funnel; a large mismatch (BFMI low) warns of trouble even when $\hat R$ looks fine.

In [ ]:
print(az.summary(idata, var_names=['mu', 'tau', 'sigma']))
n_div = int(idata.sample_stats['diverging'].sum())
print(f'divergences: {n_div}')

In [ ]:
az.plot_energy(idata); plt.tight_layout()

In [ ]:
az.plot_trace(idata, var_names=['mu', 'tau', 'sigma']); plt.tight_layout()

**The funnel, visualized.** Plot $\log\tau$ against a single group offset $z_0$. In the *non-centered* space this is a clean, roughly round cloud — no neck — which is exactly why the sampler succeeds. (The broken notebook shows the centered version, where this same pair forms the divergent funnel.)

In [ ]:
az.plot_pair(idata, var_names=['tau', 'z'], coords={'group':[0]},
             divergences=True)
plt.tight_layout()

## Step 6 — Posterior predictive checks

Does the fitted model reproduce the observed spread of the data? We overlay posterior-predictive datasets on the observed distribution; a good fit envelopes the data without systematic gaps.

In [ ]:
az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

## Step 7 — Shrinkage: the heart of partial pooling

Compare each group's **no-pooling** empirical mean to its **partial-pooling** posterior mean. Partial pooling pulls every group toward the grand mean $\hat\mu$, and pulls *small/noisy* groups more. This is the bias–variance trade made explicit: a little bias toward the population buys a large variance reduction, which is why partial pooling beats both extremes when groups have few observations.

In [ ]:
theta_post = idata.posterior['theta'].mean(dim=('chain','draw')).values
mu_hat = float(idata.posterior['mu'].mean())
fig, ax = plt.subplots(figsize=(6,4))
for j in range(J):
    ax.plot([0,1], [emp[j], theta_post[j]], color='grey', alpha=0.6)
ax.scatter(np.zeros(J), emp, color='#C44E52', label='no pooling (empirical)')
ax.scatter(np.ones(J), theta_post, color='#4C72B0', label='partial pooling')
ax.axhline(mu_hat, color='k', ls='--', lw=1, label='grand mean (mu_hat)')
ax.set(xticks=[0,1], xticklabels=['no pool','partial'],
       ylabel='group mean', title='Shrinkage toward the grand mean')
ax.legend(); plt.tight_layout()

## Step 8 — Decision & communication

Report the population parameters with uncertainty and a recovery check against the known truth. For a collaborator, the headline is the grand mean $\mu$ and the between-group SD $\tau$ (how much plates genuinely differ).

In [ ]:
from shared.bayes_utils import check_recovery
for res in check_recovery(idata, data['truth']):
    print(res)

**Conclusion (for a collaborator).** The population mean readout is ~5.0 with a tight interval; plates differ from one another with a between-plate SD $\tau\approx0.8$ — real but modest variation. Per-plate estimates should be the *shrunken* partial-pooling means, not the raw averages, especially for plates with few wells. See `summary_onepager.md`.